In [ ]:
from google.colab import files

In [ ]:
up = files.upload()

Saving train_baseline.csv to train_baseline.csv
Saving train_heavytail_diurnal.csv to train_heavytail_diurnal.csv


In [23]:
import pandas as pd

train = pd.read_csv('train_baseline.csv')
heldout = pd.read_csv('train_heavytail_diurnal.csv')

for d in (train, heldout):
    # fast/slow ratio: <1 means rate just dropped (entering quiet -> sleep safe);
    # >1 means rate rising (burst -> keep polling)
    d['rate_ratio'] = d['rate_fast'] / (d['rate_slow'] + 1e-6)

print("train:", train.shape, "| heldout:", heldout.shape)
print(train['best_sleep'].value_counts(normalize=True).sort_index())

train: (199317, 7) | heldout: (198365, 7)
best_sleep
0      0.200986
20     0.498924
100    0.245217
250    0.054872
Name: proportion, dtype: float64


In [24]:
FEATURES = ['empty_run','us_since_pkt','rate_fast','rate_med','rate_slow', 'rate_ratio']
LABEL = 'best_sleep'

X = train[FEATURES];  y = train[LABEL]

In [25]:
from sklearn.model_selection import train_test_split

# Hold out 25% of the BASELINE data to measure accuracy on rows the model
# didn't train on. stratify=y keeps the same class proportions in both halves.
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)

print("train rows:", len(X_tr), " test rows:", len(X_te))

train rows: 149487  test rows: 49830


In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# max_iter bumped up so the solver fully converges on 6 features x 4 classes.
logreg = LogisticRegression(max_iter=2000)
logreg.fit(X_tr, y_tr)

pred = logreg.predict(X_te)
print(f"accuracy: {accuracy_score(y_te, pred):.3f}")

# Per-class precision/recall
print(classification_report(y_te, pred, digits=3, zero_division=0))

accuracy: 0.513
              precision    recall  f1-score   support

           0      0.603     0.174     0.270     10015
          20      0.508     0.959     0.664     24862
         100      0.000     0.000     0.000     12219
         250      0.000     0.000     0.000      2734

    accuracy                          0.513     49830
   macro avg      0.278     0.283     0.233     49830
weighted avg      0.375     0.513     0.385     49830



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [27]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, confusion_matrix

# max_depth=6 keeps the tree small
tree = DecisionTreeClassifier(max_depth=6, random_state=0)
tree.fit(X_tr, y_tr)

pred_t = tree.predict(X_te)
print(f"accuracy: {accuracy_score(y_te, pred_t):.3f}")

# feature_importances_: how much each feature contributed to the tree's splits.
# We EXPECT rate_fast (and maybe rate_med) to dominate.
# If empty_run/us_since_pkt dominate instead, the EWMAs didn't help.
print("\nfeature importances:")
for name, imp in zip(FEATURES, tree.feature_importances_):
    print(f"  {name:14s} {imp:.3f}")

# The top of the tree in plain English
print("\ntop splits:")
print(export_text(tree, feature_names=FEATURES, max_depth=2))

accuracy: 0.528

feature importances:
  empty_run      0.007
  us_since_pkt   0.003
  rate_fast      0.000
  rate_med       0.216
  rate_slow      0.770
  rate_ratio     0.004

top splits:
|--- rate_slow <= 0.01
|   |--- rate_med <= 0.01
|   |   |--- rate_slow <= 0.01
|   |   |   |--- truncated branch of depth 4
|   |   |--- rate_slow >  0.01
|   |   |   |--- truncated branch of depth 4
|   |--- rate_med >  0.01
|   |   |--- rate_slow <= 0.01
|   |   |   |--- truncated branch of depth 4
|   |   |--- rate_slow >  0.01
|   |   |   |--- truncated branch of depth 4
|--- rate_slow >  0.01
|   |--- rate_med <= 0.00
|   |   |--- rate_med <= 0.00
|   |   |   |--- truncated branch of depth 4
|   |   |--- rate_med >  0.00
|   |   |   |--- truncated branch of depth 4
|   |--- rate_med >  0.00
|   |   |--- rate_med <= 0.01
|   |   |   |--- truncated branch of depth 4
|   |   |--- rate_med >  0.01
|   |   |   |--- truncated branch of depth 4



In [28]:
import numpy as np

# Rows = true class, columns = predicted class. The DIAGONAL is correct
# predictions; OFF-diagonal is mistakes. The cell at (true=100, pred=250)
# and (true=250, pred=100) is the confusion we predicted -- those two classes
# look almost identical in feature space because "how deep into a quiet period
# you are" is memoryless. A big number there is a FINDING, not a failure.
classes = [0, 20, 100, 250]
cm = confusion_matrix(y_te, pred_t, labels=classes)

print("confusion matrix (rows=true, cols=pred):")
print("        " + "".join(f"{c:>8}" for c in classes))
for i, c in enumerate(classes):
    print(f"true {c:>4} " + "".join(f"{cm[i][j]:>8}" for j in range(len(classes))))

confusion matrix (rows=true, cols=pred):
               0      20     100     250
true    0     4602    5411       2       0
true   20     3140   21713       9       0
true  100      438   11775       6       0
true  250      109    2625       0       0


In [29]:
# How does the tree do on traffic it NEVER trained on
Xh = heldout[FEATURES]
yh = heldout[LABEL]
print(f"held-out (heavy-tail+diurnal) accuracy: {tree.score(Xh, yh):.3f}")
print("\nclassification on held-out traffic:")
print(classification_report(yh, tree.predict(Xh), digits=3))

held-out (heavy-tail+diurnal) accuracy: 0.598

classification on held-out traffic:
              precision    recall  f1-score   support

           0      0.678     0.570     0.619     50772
          20      0.577     0.877     0.696    102315
         100      0.257     0.001     0.003     39293
         250      0.000     0.000     0.000      5985

    accuracy                          0.598    198365
   macro avg      0.378     0.362     0.330    198365
weighted avg      0.522     0.598     0.518    198365



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [30]:
# Recode to binary: the 4-class model only ever used 2 actions, with zero
# recall on the two long sleeps across BOTH datasets -> sleep DEPTH is not
# learnable (memoryless quiet periods). Reduce to the learnable decision.
for d in (train, heldout):
    d['sleep'] = (d['best_sleep'] > 0).astype(int)   # 1 = sleep, 0 = keep polling

LABEL = 'sleep'

# check the new majority floor -- this is the bar the model must beat
print(train[LABEL].value_counts(normalize=True))


sleep
1    0.799014
0    0.200986
Name: proportion, dtype: float64


In [31]:
from sklearn.model_selection import train_test_split
X = train[FEATURES]; y = train[LABEL]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)

In [32]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

tree = DecisionTreeClassifier(max_depth=6, random_state=0)   # no class_weight
tree.fit(X_tr, y_tr)
pred = tree.predict(X_te)

print(f"test accuracy: {accuracy_score(y_te, pred):.3f}")
print(classification_report(y_te, pred, digits=3, zero_division=0))

print("feature importances:")
for nm, im in zip(FEATURES, tree.feature_importances_):
    print(f"  {nm:14s} {im:.3f}")

print("\nconfusion (rows=true, cols=pred):  [0=poll, 1=sleep]")
cm = confusion_matrix(y_te, pred, labels=[0,1])
print("        pred0   pred1")
for i,c in enumerate([0,1]):
    print(f"true {c}  {cm[i][0]:>7} {cm[i][1]:>7}")

# held-out generalization
print(f"\nheld-out accuracy: {tree.score(heldout[FEATURES], heldout[LABEL]):.3f}")
print(classification_report(heldout[LABEL], tree.predict(heldout[FEATURES]), digits=3, zero_division=0))

test accuracy: 0.823
              precision    recall  f1-score   support

           0      0.589     0.388     0.468     10015
           1      0.858     0.932     0.894     39815

    accuracy                          0.823     49830
   macro avg      0.724     0.660     0.681     49830
weighted avg      0.804     0.823     0.808     49830

feature importances:
  empty_run      0.003
  us_since_pkt   0.003
  rate_fast      0.001
  rate_med       0.220
  rate_slow      0.770
  rate_ratio     0.003

confusion (rows=true, cols=pred):  [0=poll, 1=sleep]
        pred0   pred1
true 0     3889    6126
true 1     2709   37106

held-out accuracy: 0.820
              precision    recall  f1-score   support

           0      0.699     0.521     0.597     50772
           1      0.848     0.923     0.884    147593

    accuracy                          0.820    198365
   macro avg      0.774     0.722     0.740    198365
weighted avg      0.810     0.820     0.811    198365



In [33]:
from sklearn.linear_model import LogisticRegression
logreg = LogisticRegression(max_iter=5000)
logreg.fit(X_tr, y_tr)
print(f"logreg test accuracy: {accuracy_score(y_te, logreg.predict(X_te)):.3f}")

logreg test accuracy: 0.810
